# Loan Performance Dashboard - Step 1: Data Cleaning
Dataset: Lending Club accepted loans (Kaggle). Output: `loans_clean.csv` for SQL Server / Power BI.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np

# Change this to where your Kaggle file is.
# Colab: upload the file (left sidebar > Files) then use "accepted_2007_to_2018Q4.csv"
# Windows: r"C:\Users\YourName\Downloads\accepted_2007_to_2018Q4.csv"
FILE_PATH = "accepted_2007_to_2018Q4.csv"

## 2. Load only the columns we need (saves memory)

In [ ]:
cols = ["loan_amnt","term","int_rate","grade","emp_length","home_ownership",
        "annual_inc","verification_status","issue_d","loan_status",
        "purpose","addr_state","dti"]

df = pd.read_csv(FILE_PATH, usecols=cols, low_memory=False)
print("Rows loaded:", len(df))
df.head()

## 3. Keep only finished loans
We can only judge default on loans that already ended.

In [ ]:
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()
print("Finished loans:", len(df))
df["loan_status"].value_counts()

## 4. Sample 150,000 rows (keeps files small and fast)

In [ ]:
if len(df) > 150000:
    df = df.sample(150000, random_state=42)
df = df.reset_index(drop=True)
len(df)

## 5. Check missing values

In [ ]:
df.isnull().sum()

In [ ]:
# Drop rows missing key fields, fill the less important ones
df = df.dropna(subset=["loan_amnt","int_rate","annual_inc","issue_d","loan_status"])
df["emp_length"] = df["emp_length"].fillna("Unknown")
df["dti"] = df["dti"].fillna(df["dti"].median())
df.isnull().sum()

## 6. Make the default flag (main KPI)
1 = Charged Off (defaulted), 0 = Fully Paid

In [ ]:
df["default"] = (df["loan_status"] == "Charged Off").astype(int)
print("Default rate: {:.1%}".format(df["default"].mean()))

## 7. Fix data types and text columns

In [ ]:
# Date
df["issue_d"] = pd.to_datetime(df["issue_d"], format="%b-%Y")
df["issue_year"] = df["issue_d"].dt.year

# term: " 36 months" -> 36
df["term"] = df["term"].astype(str).str.extract(r"(\d+)").astype(int)

# int_rate: sometimes "13.5%" text, sometimes already a number
if df["int_rate"].dtype == "object":
    df["int_rate"] = df["int_rate"].str.replace("%", "").astype(float)

# Clean text
for c in ["grade","home_ownership","verification_status","purpose","addr_state"]:
    df[c] = df[c].astype(str).str.strip()

df["purpose"] = df["purpose"].str.replace("_", " ").str.title()
df.dtypes

## 8. Remove outliers

In [ ]:
before = len(df)
df = df[(df["annual_inc"] > 0) & (df["annual_inc"] <= df["annual_inc"].quantile(0.99))]
df = df[(df["dti"] >= 0) & (df["dti"] <= 60)]
print("Removed", before - len(df), "outlier rows")
df.describe()

## 9. Add helpful columns for the dashboard

In [ ]:
# Income bands
df["income_band"] = pd.cut(
    df["annual_inc"],
    bins=[0, 30000, 60000, 100000, 150000, np.inf],
    labels=["<30k", "30-60k", "60-100k", "100-150k", "150k+"]
)

# Loan size bands
df["loan_band"] = pd.cut(
    df["loan_amnt"],
    bins=[0, 5000, 10000, 20000, 30000, np.inf],
    labels=["<5k", "5-10k", "10-20k", "20-30k", "30k+"]
)
df.head()

## 10. Quick sanity checks (also good insights for your README)

In [ ]:
print(df.groupby("grade")["default"].mean().round(3))
print()
print(df.groupby("purpose")["default"].mean().sort_values(ascending=False).round(3))
print()
print(df.groupby("issue_year")["default"].mean().round(3))

## 11. Save the clean file

In [ ]:
df.to_csv("loans_clean.csv", index=False)
print("Saved loans_clean.csv with", len(df), "rows")

# If you are on Colab, uncomment to download:
# from google.colab import files
# files.download("loans_clean.csv")